# Meta-Analysis Script for Genetic Association Studies (Including Egypt vs Sham Analysis)

This Python script performs a meta-analysis of genetic data from multiple datasets (e.g., Sham and Egypt), combining results from individual studies to estimate pooled effect sizes, assess heterogeneity, and generate various plots to summarize the findings. Additionally, the script includes data visualization comparing average odds ratios (OR) by country and scatter plots of genes associated with Type 2 Diabetes Mellitus (T2DM) in both datasets.

## Full Code with Explanations

### 1. Import Necessary Libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import argparse
import logging
from datetime import datetime
from statsmodels.stats.meta_analysis import combine_effects
import statsmodels.api as sm
from scipy.stats import chi2
Explanation:

Imports: Libraries for data manipulation (Pandas, NumPy), statistical analysis (Statsmodels), visualization (Matplotlib, Seaborn), and system utilities (logging, os) are imported here.

Logging: This is used for debugging and tracking script execution, which helps collaborators to debug and understand the flow of the script.

2. Argument Parsing and Directory Setup

def setup_args():
    """Setup input/output directories and handle command line arguments."""
    parser = argparse.ArgumentParser(description='Meta-Analysis for genetic data')
    parser.add_argument('--input', type=str, default='sham_data1.xlsx', help='Input Excel file')
    parser.add_argument('--output', type=str, default='meta_analysis_results', help='Results directory')
    args, _ = parser.parse_known_args()
    
    results_dir = args.output
    os.makedirs(results_dir, exist_ok=True)
    os.makedirs(os.path.join(results_dir, 'eda'), exist_ok=True)
    os.makedirs(os.path.join(results_dir, 'meta_analysis'), exist_ok=True)
    
    logging.info(f"Output directory created at {results_dir}")
    return args.input, results_dir



Explanation:

setup_args(): Handles command-line arguments for input files and output directory setup. This makes the script versatile for use with different datasets.

3. Load and Clean Data


def load_and_clean_data(file_path):
    """Load and clean input data."""
    df = pd.read_excel(file_path)
    df.columns = df.columns.str.strip()  # Strip any extra spaces from column names
    df = df.rename(columns={
        'OR': 'OR', '95% CI': 'CI', 'Gene': 'gene', 'SNP': 'snp', 'Allele': 'allele',
        'Country': 'country', 'No. of Patients': 'patients', 'No. of controls': 'controls',
        'Genotype Method': 'method', 'P-Value': 'p_value', 'Reference': 'reference'
    })
    
    # Cleaning CI and OR data
    df['CI'] = df['CI'].str.replace(r'[–—−\s]', '-', regex=True)
    df['CI_Lower'] = df['CI'].str.extract(r'^([0-9.]+)')[0].astype(float)
    df['CI_Upper'] = df['CI'].str.extract(r'-([0-9.]+)$')[0].astype(float)
    df['OR'] = df['OR'].astype(float)
    df['logOR'] = np.where(df['OR'] > 0, np.log(df['OR']), np.nan)
    df['SE'] = np.where((df['CI_Lower'] > 0) & (df['CI_Upper'] < np.inf),
                        (np.log(df['CI_Upper']) - np.log(df['CI_Lower'])) / (2 * 1.96), np.nan)
    df['p_value'] = df['p_value'].astype(float)
    
    df_meta = df.dropna(subset=['logOR', 'SE', 'OR', 'CI_Lower', 'CI_Upper', 'p_value'])
    if len(df_meta) == 0:
        logging.error('No valid rows found in the data. Ensure the dataset contains proper values.')
        raise ValueError('Data is missing required values.')
    
    logging.info(f"Data loaded and cleaned. {len(df_meta)} valid rows.")
    return df_meta



def load_and_clean_data(file_path):
    """Load and clean input data."""
    df = pd.read_excel(file_path)
    df.columns = df.columns.str.strip()  # Strip any extra spaces from column names
    df = df.rename(columns={
        'OR': 'OR', '95% CI': 'CI', 'Gene': 'gene', 'SNP': 'snp', 'Allele': 'allele',
        'Country': 'country', 'No. of Patients': 'patients', 'No. of controls': 'controls',
        'Genotype Method': 'method', 'P-Value': 'p_value', 'Reference': 'reference'
    })
    
    # Cleaning CI and OR data
    df['CI'] = df['CI'].str.replace(r'[–—−\s]', '-', regex=True)
    df['CI_Lower'] = df['CI'].str.extract(r'^([0-9.]+)')[0].astype(float)
    df['CI_Upper'] = df['CI'].str.extract(r'-([0-9.]+)$')[0].astype(float)
    df['OR'] = df['OR'].astype(float)
    df['logOR'] = np.where(df['OR'] > 0, np.log(df['OR']), np.nan)
    df['SE'] = np.where((df['CI_Lower'] > 0) & (df['CI_Upper'] < np.inf),
                        (np.log(df['CI_Upper']) - np.log(df['CI_Lower'])) / (2 * 1.96), np.nan)
    df['p_value'] = df['p_value'].astype(float)
    
    df_meta = df.dropna(subset=['logOR', 'SE', 'OR', 'CI_Lower', 'CI_Upper', 'p_value'])
    if len(df_meta) == 0:
        logging.error('No valid rows found in the data. Ensure the dataset contains proper values.')
        raise ValueError('Data is missing required values.')
    
    logging.info(f"Data loaded and cleaned. {len(df_meta)} valid rows.")
    return df_meta
Explanation:

Data Cleaning: This function loads the input dataset and processes it, ensuring that columns are correctly named and missing or invalid data are handled properly.


4. Exploratory Data Analysis (EDA) Plots

def generate_eda_plots(df_meta, results_dir):
    """Generate exploratory data analysis plots."""
    sns.set(style='whitegrid', font_scale=1.2)
    
    # Bar Plot: Studies per Gene
    genes = df_meta['gene'].value_counts().reset_index()
    genes.columns = ['gene', 'count']
    plt.figure(figsize=(8, max(6, len(genes) * 0.3)))
    sns.barplot(data=genes, x='count', y='gene', color='#0072B2')
    plt.title('Number of Studies per Gene', fontsize=16, pad=15)
    plt.xlabel('Number of Studies', fontsize=12)
    plt.ylabel('Gene', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'eda', 'studies_per_gene.png'), dpi=350, bbox_inches='tight')
    plt.close()

    logging.info("EDA plots generated and saved.")
Explanation:

Bar Plot: Visualizes the number of studies conducted for each gene, providing an overview of the distribution of research efforts.

5. Average Odds Ratio (OR) Comparison Between Egypt and Sham Datasets

# --- LOAD DATA ---
sham = pd.read_excel('sham_data1.xlsx')
egypt = pd.read_excel('egypt_data.xlsx')

# =============================
# PLOT 1: AVERAGE OR BY COUNTRY (SIDE-BY-SIDE BARPLOT)
# =============================
def get_avg_or(df):
    df = df.copy()
    df['OR'] = pd.to_numeric(df['OR'], errors='coerce')
    country_avg = df.groupby('Country', as_index=False)['OR'].mean()
    return country_avg.dropna()

sham_avg = get_avg_or(sham)
egypt_avg = get_avg_or(egypt)

# Combine, label origin
egypt_avg['Group'] = 'Egypt dataset'
sham_avg['Group'] = 'Sham dataset'
all_avg = pd.concat([egypt_avg, sham_avg], ignore_index=True)

# Pivot for grouped barplot
df_pivot = all_avg.pivot(index='Country', columns='Group', values='OR').fillna(0)

# --- Plot ---
sns.set(style="whitegrid", font_scale=1.25)
ax = df_pivot.plot(kind='bar', figsize=(8,5), width=0.7, rot=0, dpi=300)
ax.set_ylabel('Average Odds Ratio (OR)')
ax.set_title('Average Odds Ratio by Country (Egypt vs. Sham datasets)')
ax.axhline(y=1, color='red', linestyle='--', linewidth=2, label='OR=1')
ax.legend()
plt.tight_layout()
plt.show()


Explanation:

Average OR Comparison: This block calculates the average odds ratio (OR) by country for both Egypt and Sham datasets and compares them side by side using a bar plot. A red dashed line at OR=1 indicates no association.

6. Scatter Plot by Gene: Egypt vs Sham

# =============================
# PLOT 2: SCATTER PLOT BY GENE (EGYPT VS SHAM)
# =============================
# Prepare data
egypt_plot = egypt[['Gene', 'OR', 'Allele']].copy()
egypt_plot['Source'] = 'Egypt'
sham_plot = sham[['Gene', 'OR', 'Allele']].copy()
sham_plot['Source'] = 'Sham'

egypt_plot['OR'] = pd.to_numeric(egypt_plot['OR'], errors='coerce')
sham_plot['OR'] = pd.to_numeric(sham_plot['OR'], errors='coerce')
egypt_plot = egypt_plot.dropna(subset=['OR'])
sham_plot = sham_plot.dropna(subset=['OR'])

full = pd.concat([egypt_plot, sham_plot], ignore_index=True)

plt.figure(figsize=(14, 6), dpi=200)
plt.scatter(egypt_plot['Gene'], egypt_plot['OR'], color='blue', marker='x', s=120, label='Egypt')
plt.scatter(sham_plot['Gene'], sham_plot['OR'], color='green', marker='x', s=120, label='Levant (Sham)')

# Annotate allele names above points
for idx, row in full.iterrows():
    plt.text(row['Gene'], row['OR'] + 0.05*full['OR'].max(), str(row['Allele']),
             fontsize=9, ha='center', color='gray')

plt.axhline(y=1, color='red', linestyle='--', linewidth=2, label='No association line (OR=1)')
plt.ylabel('Odds Ratio (OR)')
plt.xlabel('Gene')
plt.title('Comparison of Alleles Associated with T2DM (Egypt vs. Sham)')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()


Explanation:

Scatter Plot: A scatter plot that compares the Odds Ratios (OR) of genetic associations by Gene between Egypt and Sham datasets. The points are annotated with allele names, and a red dashed line at OR=1 indicates no association.

7. Meta-Analysis

def meta_analysis(df_meta, results_dir):
    """Perform meta-analysis for each SNP."""
    snp_counts = df_meta['snp'].value_counts()
    multi_study_snps = snp_counts[snp_counts >= 2].index
    summary_out = []
    
    for snp in multi_study_snps:
        df_snp = df_meta[df_meta['snp'] == snp]
        gene_name = df_snp['gene'].iloc[0]
        effects = df_snp['logOR'].values
        ses = df_snp['SE'].values
        studies = df_snp['study_id'].values
        
        try:
            # Random effects meta-analysis
            random = combine_effects(effects, ses, method_re='iterated')
            pooled_logor = random.effect[0] if isinstance(random.effect, np.ndarray) else random.effect
            pooled_se = random.sd_eff[0] if isinstance(random.sd_eff, np.ndarray) else random.sd_eff
            pooled_or = np.exp(pooled_logor)
            pooled_ci = np.exp([pooled_logor - 1.96 * pooled_se, pooled_logor + 1.96 * pooled_se])
            Q = random.q
            df_Q = random.k - 1
            p_Q = 1 - chi2.cdf(Q, df_Q)
            I2 = max(0, random.i2) * 100
            tau2 = random.tau2
            
            summary_out.append({
                'SNP': snp, 'Gene': gene_name, 'n_studies': len(effects),
                'Pooled_OR': pooled_or, 'Lower_95CI': pooled_ci[0], 'Upper_95CI': pooled_ci[1],
                'I2': I2, 'tau2': tau2, 'pval': p_Q
            })
            
            logging.info(f"Meta-analysis completed for SNP {snp}.")
        
        except Exception as e:
            logging.error(f"Error processing SNP {snp}: {str(e)}")
    
    if summary_out:
        summary_df = pd.DataFrame(summary_out)
        summary_df.to_csv(os.path.join(results_dir, 'meta_summary_overview.csv'), index=False)
        logging.info("Meta-analysis results saved.")
    else:
        logging.warning("No results to save for meta-analysis.")


Explanation:

Meta-Analysis: This part runs the meta-analysis for SNPs that appear in multiple studies. It computes pooled odds ratios and confidence intervals using a random-effects model, assesses heterogeneity (I², Tau²), and saves the results.

8. Main Function: Bringing It All Together


def main():
    """Main function to execute the entire pipeline."""
    input_file, results_dir = setup_args()
    
    try:
        df_meta = load_and_clean_data(input_file)
        generate_eda_plots(df_meta, results_dir)
        meta_analysis(df_meta, results_dir)
        logging.info("Meta-analysis complete. See results in the output directory.")
    except Exception as e:
        logging.error(f"Error in meta-analysis pipeline: {str(e)}")

if __name__ == '__main__':
    main()
Explanation:

main(): Orchestrates the entire analysis pipeline. It runs the functions to load data, generate exploratory plots, and perform the meta-analysis.

Final Notes:
This script is now ready for use in collaborative environments. By generating both exploratory plots (e.g., for average OR comparisons) and statistical analyses (e.g., meta-analysis), it provides a comprehensive view of genetic associations across datasets.

Run the Script:
Ensure the necessary datasets (sham_data1.xlsx and egypt_data.xlsx) are present and properly formatted.

Command Line:
You can run the script with customized input and output directories.



